In [ ]:
print(df.head())

               0          1           2                 3          4    5   \
0         burgers  meatballs        eggs               NaN        NaN  NaN   
1         chutney        NaN         NaN               NaN        NaN  NaN   
2          turkey    avocado         NaN               NaN        NaN  NaN   
3   mineral water       milk  energy bar  whole wheat rice  green tea  NaN   
4  low fat yogurt        NaN         NaN               NaN        NaN  NaN   

    6    7    8    9    10   11   12   13   14   15   16   17   18   19  
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  


In [ ]:
import pandas as pd

# ১. ডেটাসেট লোড ও ট্রানজেকশন প্রসেসিং
df = pd.read_csv('Market_Basket_Optimisation.csv', header=None)
if df.iloc[0, 0] == 'Item1':
    df = df.iloc[1:].reset_index(drop=True)

transactions = [set(df.iloc[i].dropna().values) for i in range(len(df))]
total_transactions = len(transactions)

# দোকানে থাকা সকল ইউনিক প্রোডাক্টের তালিকা
all_items = sorted(list(set([item for tx in transactions for item in tx])))

# একক ও যৌথ ক্রয়ের গণনা (Pre-computing Counts)
item_counts = {item: 0 for item in all_items}
co_counts = {item: {other: 0 for other in all_items} for item in all_items}

for tx in transactions:
    for item in tx:
        item_counts[item] += 1
        for other in tx:
            if item != other:
                co_counts[item][other] += 1

# মূল প্রেডিকশন ফাংশন
def predict_all_products(selected_product):
    product_map = {item.lower(): item for item in all_items}

    if selected_product.lower() not in product_map:
        print(f"\n '{selected_product}' পণ্যটি দোকানে খুঁজে পাওয়া যায়নি। দয়া করে সঠিক বানান লিখুন।\n")
        return

    actual_product = product_map[selected_product.lower()]
    target_count = item_counts[actual_product]

    results = []
    for other_item in all_items:
        if other_item == actual_product:
            continue

        co_occur = co_counts[actual_product][other_item]

        # Support (%) calculation
        support = (co_occur / total_transactions) * 100

        # Confidence (%) calculation (Buying Probability)
        confidence = (co_occur / target_count) * 100

        # Lift calculation
        prob_B = item_counts[other_item] / total_transactions
        lift = (co_occur / target_count) / prob_B if prob_B > 0 else 0

        results.append({
            'Shop Product': other_item,
            'Co-occurrence (Times)': co_occur,
            'Support (%)': round(support, 2),
            'Buying Probability / Confidence (%)': round(confidence, 2),
            'Lift': round(lift, 2)
        })

    # সম্ভাবনা (Confidence) অনুসারে সর্বোচ্চ থেকে সর্বনিম্ন সাজানো
    res_df = pd.DataFrame(results).sort_values(by='Buying Probability / Confidence (%)', ascending=False).reset_index(drop=True)

    print(f"\n-------------------------------------------------------------")
    print(f" কাস্টমার '{actual_product}' কিনলে অন্য সব প্রোডাক্ট কেনার সম্ভাবনা (Apriori Analysis):")
    print(f" (মোট ক্রয়: {target_count} বার | মোট অন্যান্য প্রোডাক্ট: {len(res_df)} টি)")
    print(f"---------------------------------------------------------------\n")

    pd.set_option('display.max_rows', None)
    print(res_df)
    print("\n")

# ইউজার ইনপুট লুপ
while True:
    user_input = input("প্রোডাক্টের নাম লিখুন (বন্ধ করতে 'exit' লিখুন): ").strip()
    if user_input.lower() == 'exit':
        print("প্রোগ্রাম বন্ধ করা হয়েছে।")
        break
    predict_all_products(user_input)

প্রোডাক্টের নাম লিখুন (বন্ধ করতে 'exit' লিখুন): pasta

-------------------------------------------------------------
 কাস্টমার 'pasta' কিনলে অন্য সব প্রোডাক্ট কেনার সম্ভাবনা (Apriori Analysis):
 (মোট ক্রয়: 118 বার | মোট অন্যান্য প্রোডাক্ট: 119 টি)
---------------------------------------------------------------

             Shop Product  Co-occurrence (Times)  Support (%)  \
0                escalope                     44         0.59   
1                  shrimp                     38         0.51   
2            french fries                     23         0.31   
3                    eggs                     21         0.28   
4    mushroom cream sauce                     20         0.27   
5               chocolate                     19         0.25   
6               green tea                     18         0.24   
7                 burgers                     17         0.23   
8           mineral water                     16         0.21   
9                    milk           